In [35]:
#import dependencies
import pandas as pd
import sqlite3
import hvplot.pandas
from pathlib import Path
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

In [36]:
#create a sqlite3 file
db_name = "opioid_analysis.sqlite"

conn = sqlite3.connect(db_name)
cursor = conn.cursor()

In [ ]:
#skip this step if the database is already created in postreSQL 
#read our cleaned data csv
#df = pd.read_csv("CT_OpioidData_Final.csv")
#df.head()

In [ ]:
#this is inserting my database into the sqlite3 file (after reading the CSV)
#df.to_sql("ct_opioiddata_final", conn, if_exists="replace", index=False)

In [37]:
#query
query = "SELECT * FROM ct_opioiddata_final;"
opioid_df = pd.read_sql(query, conn)

In [38]:
#inspect to make sure we imported and created it properly
print(opioid_df.head())

   IncidentID DateReported   Sex                       Race  Age  \
0           1       1/1/19  Male                      White   58   
1           2       1/1/19  Male  Black or African American   22   
2           3       1/1/19  Male            Hispanic, White   60   
3           4       1/1/19  Male            Hispanic, White   33   
4           5       1/2/19  Male                      White   24   

                      InjuryPlace   InjuryCity InjuryCounty InjuryState  \
0                       Residence   Bridgeport    Fairfield          CT   
1  Other, Residential Institution      Norwalk    Fairfield          CT   
2                       Residence    Waterbury    New Haven          CT   
3                       Residence      Meriden    New Haven          CT   
4                       Residence  Glastonbury     Hartford          CT   

   InjuryZip  Heroin  Cocaine  Fetanyl  Fentanyl Analogue  Oxycodone  \
0       6604       1        1        0                  0          0

In [39]:
#inspect column names
print(opioid_df.columns)

Index(['IncidentID', 'DateReported', 'Sex', 'Race', 'Age', 'InjuryPlace',
       'InjuryCity', 'InjuryCounty', 'InjuryState', 'InjuryZip', 'Heroin',
       'Cocaine', 'Fetanyl', 'Fentanyl Analogue', 'Oxycodone', 'Oxymorphone',
       'Ethanol', 'Hydrocodone', 'Benzodiazepines', 'Methadone',
       'Meth/Amphetamine', 'Tramadol', 'Hydromorphone',
       'Morphine (not heroin)', 'Other', 'Unnamed: 25', 'Unnamed: 26'],
      dtype='object')


In [ ]:
#filter certain columns that would apply to my visualization: # deaths by drug type
#columns_to_filter = ["Heroin", "Cocaine", "Fetanyl", "Fentanyl Analogue", "Oxycodone", "Oxymorphone", "Ethanol", "Hydrocodone", "Benzodiazepines", "Methadone", "Meth/Amphetamine", "Tramadol", "Hydromorphone", "Morphine (not heroin)", "Other"]

In [ ]:
#create a loop for existing columns and the columns we want to filter
#existing_columns = [col for col in columns_to_filter if col in opioid_df.columns]

In [ ]:
#create a dataframe from the filtered columns
#carleigh_filtered_df = opioid_df[existing_columns]
#carleigh_filtered_df.head()

In [ ]:
#filter certain columns that would apply to my visualization: # deaths by drug type
#opioid_df = opioid_df.drop(columns=["IncidentID", "InjuryCity", "InjuryCounty", "InjuryState", "InjuryState, "Unnamed: 25", "Unnamed: 26", "DateReported"])

In [ ]:
#from sklearn.preprocessing import LabelEncoder

In [40]:
opioid_df = opioid_df.drop(columns=["Age", "IncidentID", "InjuryCity", "InjuryZip", "Other", "InjuryCounty", "InjuryState", "Unnamed: 25", "Unnamed: 26", "DateReported"])

In [41]:
print(opioid_df.dtypes)

Sex                      object
Race                     object
InjuryPlace              object
Heroin                    int64
Cocaine                   int64
Fetanyl                   int64
Fentanyl Analogue         int64
Oxycodone                 int64
Oxymorphone               int64
Ethanol                   int64
Hydrocodone               int64
Benzodiazepines           int64
Methadone                 int64
Meth/Amphetamine          int64
Tramadol                  int64
Hydromorphone             int64
Morphine (not heroin)     int64
dtype: object


In [42]:
#count how many places there are in InjuryPlace
injury_place_counts = opioid_df["InjuryPlace"].value_counts()
injury_place_counts

InjuryPlace
Residence                 3263
Home                      1781
Other                      523
Other Specified Place      367
Unknown                    363
                          ... 
Railroad Track               1
River, Stream or Canal       1
Relative's Home              1
residence                    1
Motel/Hotel                  1
Name: count, Length: 87, dtype: int64

In [43]:
#consolidating the InjuryPlace column with a threshhold
injury_place_to_replace = injury_place_counts[injury_place_counts < 100].index

for injury in injury_place_to_replace:
    opioid_df["InjuryPlace"] = opioid_df["InjuryPlace"] .replace(injury, "Other")

In [44]:
#one-hot encode selected columns
opioid_dummies = pd.get_dummies(opioid_df)

In [45]:
print(opioid_dummies.columns)

Index(['Heroin', 'Cocaine', 'Fetanyl', 'Fentanyl Analogue', 'Oxycodone',
       'Oxymorphone', 'Ethanol', 'Hydrocodone', 'Benzodiazepines', 'Methadone',
       'Meth/Amphetamine', 'Tramadol', 'Hydromorphone',
       'Morphine (not heroin)', 'Sex_Female', 'Sex_Male', 'Sex_Unknown',
       'Sex_X', 'Race_American Indian or Alaska Native', 'Race_Asian',
       'Race_Asian/Indian', 'Race_Black or African American',
       'Race_Hispanic, Black', 'Race_Hispanic, White', 'Race_Other',
       'Race_Unknown', 'Race_White', 'Race_white', 'InjuryPlace_Home',
       'InjuryPlace_Hotel or Motel', 'InjuryPlace_House',
       'InjuryPlace_In Vehicle', 'InjuryPlace_Other',
       'InjuryPlace_Other Specified Place', 'InjuryPlace_Residence',
       'InjuryPlace_Unknown', 'InjuryPlace_Unspecified Place'],
      dtype='object')


In [46]:
# sanity check
print(opioid_dummies.dtypes)

Heroin                                   int64
Cocaine                                  int64
Fetanyl                                  int64
Fentanyl Analogue                        int64
Oxycodone                                int64
Oxymorphone                              int64
Ethanol                                  int64
Hydrocodone                              int64
Benzodiazepines                          int64
Methadone                                int64
Meth/Amphetamine                         int64
Tramadol                                 int64
Hydromorphone                            int64
Morphine (not heroin)                    int64
Sex_Female                                bool
Sex_Male                                  bool
Sex_Unknown                               bool
Sex_X                                     bool
Race_American Indian or Alaska Native     bool
Race_Asian                                bool
Race_Asian/Indian                         bool
Race_Black or

In [53]:
# Create a a list to store inertia values
inertia = []

# Create a a list to store the values of k
k = list(range(1, 11))

In [54]:
# Create a for-loop where each value of k is evaluated using the K-means algorithm
# Fit the model using the spread_df DataFrame
# Append the value of the computed inertia from the `inertia_` attribute of the KMeans model instance
for i in k:
    k_model = KMeans(n_clusters=i, random_state=1)
    k_model.fit(opioid_dummies)
    inertia.append(k_model.inertia_)

In [55]:
# Create a Dictionary that holds the list values for k and inertia
elbow_data = {"k": k, "inertia": inertia}

# Create a DataFrame using the elbow_data Dictionary
df_elbow = pd.DataFrame(elbow_data)

# Review the DataFrame
df_elbow.head()

,k,inertia
0,1,21521.271709
1,2,18570.362040
2,3,16902.492602
3,4,15744.555164
4,5,14935.303201


In [56]:
# Plot the DataFrame
df_elbow.hvplot.line(
    x="k",
    y="inertia",
    title="Elbox Curve",
    xticks=k
)

:Curve   [k]   (inertia)

In [30]:
pd.set_option('display.max_columns', None)

In [57]:
#define the model with 7 clusters
model = KMeans(n_clusters=7, random_state=3)

#fit the model
model.fit(opioid_dummies)

#make predictions
k_7 = model.predict(opioid_dummies)

#create a copy of the preprocessed data
opioid_predictions_df = opioid_dummies.copy()

#add a class column with the labels
opioid_predictions_df['cluster'] = k_7

cluster_0 = opioid_predictions_df[opioid_predictions_df['cluster'] == 0]
cluster_1 = opioid_predictions_df[opioid_predictions_df['cluster'] == 1]
cluster_2 = opioid_predictions_df[opioid_predictions_df['cluster'] == 2]
cluster_3 = opioid_predictions_df[opioid_predictions_df['cluster'] == 3]
cluster_4 = opioid_predictions_df[opioid_predictions_df['cluster'] == 4]
cluster_5 = opioid_predictions_df[opioid_predictions_df['cluster'] == 5]
cluster_6 = opioid_predictions_df[opioid_predictions_df['cluster'] == 6]

#group the clusters so we can compare/contrast
#cluster_description = opioid_predictions_df.groupby("cluster").describe()

#print results
#cluster_description

In [58]:
cluster_1
#white and/or hispanic males who died at a "residence" with cocaine and/or heroin in their system

,Heroin,Cocaine,Fetanyl,Fentanyl Analogue,Oxycodone,Oxymorphone,Ethanol,Hydrocodone,Benzodiazepines,Methadone,Meth/Amphetamine,Tramadol,Hydromorphone,Morphine (not heroin),Sex_Female,Sex_Male,Sex_Unknown,Sex_X,Race_American Indian or Alaska Native,Race_Asian,Race_Asian/Indian,Race_Black or African American,"Race_Hispanic, Black","Race_Hispanic, White",Race_Other,Race_Unknown,Race_White,Race_white,InjuryPlace_Home,InjuryPlace_Hotel or Motel,InjuryPlace_House,InjuryPlace_In Vehicle,InjuryPlace_Other,InjuryPlace_Other Specified Place,InjuryPlace_Residence,InjuryPlace_Unknown,InjuryPlace_Unspecified Place,cluster
0,1,1,0,0,0,0,1,0,0,0,0,0,0,0,False,True,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,True,False,False,1
2,1,0,0,0,1,0,0,0,0,0,0,0,1,0,False,True,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,True,False,False,1
3,0,0,1,0,0,0,0,0,0,0,0,0,0,0,False,True,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,True,False,False,1
6,1,0,1,0,0,0,0,0,0,0,0,0,0,0,False,True,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,True,False,False,1
9,1,0,0,0,0,0,1,0,0,0,0,0,0,0,False,True,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,True,False,False,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4893,0,1,0,0,0,0,0,0,0,0,0,0,0,0,False,True,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,True,False,False,1
4903,0,1,0,0,0,0,1,0,0,0,0,0,0,0,False,True,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,True,False,False,1
4905,0,1,0,0,0,0,1,0,0,0,0,0,0,0,False,True,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,True,False,False,1
4906,0,1,0,0,0,0,0,0,0,0,0,0,0,0,False,True,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,True,False,False,1


In [60]:
cluster_2
#white males died at home with either fetanyl or cocaine in their system, sometimes both

,Heroin,Cocaine,Fetanyl,Fentanyl Analogue,Oxycodone,Oxymorphone,Ethanol,Hydrocodone,Benzodiazepines,Methadone,Meth/Amphetamine,Tramadol,Hydromorphone,Morphine (not heroin),Sex_Female,Sex_Male,Sex_Unknown,Sex_X,Race_American Indian or Alaska Native,Race_Asian,Race_Asian/Indian,Race_Black or African American,"Race_Hispanic, Black","Race_Hispanic, White",Race_Other,Race_Unknown,Race_White,Race_white,InjuryPlace_Home,InjuryPlace_Hotel or Motel,InjuryPlace_House,InjuryPlace_In Vehicle,InjuryPlace_Other,InjuryPlace_Other Specified Place,InjuryPlace_Residence,InjuryPlace_Unknown,InjuryPlace_Unspecified Place,cluster
4347,0,0,1,0,0,0,1,0,0,0,0,0,0,0,False,True,False,False,False,False,False,False,False,False,False,False,True,False,True,False,False,False,False,False,False,False,False,2
4354,0,0,1,0,0,0,0,0,0,0,0,0,0,0,False,True,False,False,False,False,False,False,False,False,False,False,True,False,True,False,False,False,False,False,False,False,False,2
4451,1,0,1,0,0,0,0,0,0,0,0,0,0,0,False,True,False,False,False,False,False,False,False,False,False,False,True,False,True,False,False,False,False,False,False,False,False,2
4474,0,0,1,0,0,0,1,0,0,0,0,0,0,0,False,True,False,False,False,False,False,False,False,False,False,False,True,False,True,False,False,False,False,False,False,False,False,2
4480,0,1,1,0,0,0,0,0,0,1,0,0,0,0,False,True,False,False,False,False,False,False,False,False,False,False,True,False,True,False,False,False,False,False,False,False,False,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7662,0,0,0,0,0,0,0,0,0,0,0,0,0,0,False,True,False,False,False,False,False,False,False,False,False,False,True,False,True,False,False,False,False,False,False,False,False,2
7663,0,1,0,0,0,0,0,0,0,0,0,0,0,0,False,True,False,False,False,False,False,False,False,False,False,False,True,False,True,False,False,False,False,False,False,False,False,2
7664,0,1,1,0,0,0,0,0,0,0,0,0,0,0,False,True,False,False,False,False,False,False,False,False,False,False,True,False,True,False,False,False,False,False,False,False,False,2
7678,0,1,0,0,0,0,0,0,0,0,0,0,0,0,False,True,False,False,False,False,False,False,False,False,False,False,True,False,True,False,False,False,False,False,False,False,False,2


In [61]:
cluster_3
#females, black or white, died at a "residence" with a combination of drugs in their system: cocainem fetanyl, ethanol, meth, benzos
#will need to examine this one further - what is the connection bewteen the females in this group besides that they died at a residence?

,Heroin,Cocaine,Fetanyl,Fentanyl Analogue,Oxycodone,Oxymorphone,Ethanol,Hydrocodone,Benzodiazepines,Methadone,Meth/Amphetamine,Tramadol,Hydromorphone,Morphine (not heroin),Sex_Female,Sex_Male,Sex_Unknown,Sex_X,Race_American Indian or Alaska Native,Race_Asian,Race_Asian/Indian,Race_Black or African American,"Race_Hispanic, Black","Race_Hispanic, White",Race_Other,Race_Unknown,Race_White,Race_white,InjuryPlace_Home,InjuryPlace_Hotel or Motel,InjuryPlace_House,InjuryPlace_In Vehicle,InjuryPlace_Other,InjuryPlace_Other Specified Place,InjuryPlace_Residence,InjuryPlace_Unknown,InjuryPlace_Unspecified Place,cluster
5,0,1,1,1,0,0,0,0,0,0,0,0,0,0,True,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,3
14,0,0,0,0,0,0,1,1,0,0,0,0,1,0,True,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,3
17,0,0,0,0,1,0,0,0,1,0,0,0,0,0,True,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,True,False,False,3
19,0,0,1,0,0,0,0,0,0,0,1,0,0,0,True,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,True,False,False,3
20,0,0,1,0,0,0,0,0,0,0,0,0,0,0,True,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,True,False,False,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4897,0,1,0,0,0,0,1,0,0,1,0,0,0,0,True,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,3
4899,0,1,0,0,0,0,1,0,1,0,1,0,0,0,True,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,True,False,False,3
4900,0,0,0,0,0,0,0,0,0,0,0,0,0,0,True,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,True,False,False,3
4910,0,0,0,0,0,0,0,0,1,0,0,0,0,0,True,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,True,False,False,3


In [62]:
cluster_4
#white males, died with benzos and/or fetanyl in their system, 

,Heroin,Cocaine,Fetanyl,Fentanyl Analogue,Oxycodone,Oxymorphone,Ethanol,Hydrocodone,Benzodiazepines,Methadone,Meth/Amphetamine,Tramadol,Hydromorphone,Morphine (not heroin),Sex_Female,Sex_Male,Sex_Unknown,Sex_X,Race_American Indian or Alaska Native,Race_Asian,Race_Asian/Indian,Race_Black or African American,"Race_Hispanic, Black","Race_Hispanic, White",Race_Other,Race_Unknown,Race_White,Race_white,InjuryPlace_Home,InjuryPlace_Hotel or Motel,InjuryPlace_House,InjuryPlace_In Vehicle,InjuryPlace_Other,InjuryPlace_Other Specified Place,InjuryPlace_Residence,InjuryPlace_Unknown,InjuryPlace_Unspecified Place,cluster
4,0,1,1,0,0,0,0,0,1,0,0,0,0,0,False,True,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,True,False,False,4
11,0,0,1,0,0,0,1,0,1,0,0,0,0,0,False,True,False,False,False,False,False,False,False,False,False,False,True,False,False,True,False,False,False,False,False,False,False,4
21,1,0,1,0,0,0,0,0,1,0,0,0,0,0,False,True,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,True,False,False,4
22,1,1,1,0,0,0,0,0,1,0,1,0,0,0,False,True,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,True,False,False,4
23,1,0,0,0,1,0,0,0,1,0,1,0,0,0,False,True,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,True,False,False,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7314,0,1,1,0,0,0,0,0,1,0,0,0,0,0,False,True,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,True,False,False,False,4
7495,0,0,1,0,0,0,0,0,1,0,0,0,0,0,False,True,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,True,False,4
7572,0,0,1,0,0,0,1,0,1,0,0,0,0,0,False,True,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,True,False,False,False,4
7574,0,0,1,0,0,0,0,0,1,0,0,0,0,0,False,True,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,True,4


In [ ]:
#filter out cluster 1 and cluster 2, put the side by side
#run a .describe()
#where are the key distinctions
